<a href="https://colab.research.google.com/github/CodeByQasim/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CodeByQasim/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [6]:

from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

file_path = "/content/drive/MyDrive/Flyrank Dataset/content_refresh_anonymized.csv"

df = pd.read_csv(file_path)



print("--- Unit of Analysis Verification ---")

total_rows = len(df)
unique_content = df["content_id"].nunique()
duplicate_count = total_rows - unique_content

print("Total rows:", total_rows)
print("Unique content IDs:", unique_content)
print("Duplicate content IDs:", duplicate_count)

if duplicate_count == 0:
    print("PASS: one row represents one content item.")
else:
    print("CHECK NEEDED: duplicate content IDs exist.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Unit of Analysis Verification ---
Total rows: 30000
Unique content IDs: 30000
Duplicate content IDs: 0
PASS: one row represents one content item.


## 1. Unit of analysis + time window

For Lane 3, one row represents **one content item**.

The dataset provides historical information using a 90-day window and two 30-day comparison windows: the last 30 days and the previous 30 days. I will use these available observation windows to describe each content item.

The purpose of this lane is to use observable content, search, and performance characteristics to identify groups of similar content items through unsupervised clustering.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [7]:
# Verify the fields planned for Lane 3

print("--- Lane 3 Field Plan ---")

feature_fields = [
    "search_volume",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "avg_position"
]

context_fields = [
    "content_id",
    "content_type",
    "main_intent",
    "content_age_days",
    "freshness_tier"
]

excluded_fields = [
    "client_id",
    "provider_used",
    "model_used"
]

print("\n--- Features ---")
for column in feature_fields:
    if column in df.columns:
        print("✓", column)
    else:
        print("✗ Missing:", column)

print("\n--- Context ---")
for column in context_fields:
    if column in df.columns:
        print("✓", column)
    else:
        print("✗ Missing:", column)

print("\n--- Excluded ---")
for column in excluded_fields:
    if column in df.columns:
        print("✓", column, "- excluded")
    else:
        print("✗ Not found:", column)

--- Lane 3 Field Plan ---

--- Features ---
✓ search_volume
✓ word_count
✓ impressions_90d
✓ clicks_90d
✓ avg_position

--- Context ---
✓ content_id
✓ content_type
✓ main_intent
✓ content_age_days
✓ freshness_tier

--- Excluded ---
✓ client_id - excluded
✓ provider_used - excluded
✓ model_used - excluded


## 2. Fields: feature / label / context / excluded

### Features

For the initial Lane 3 clustering analysis, I will use five features:

1. **search_volume** — represents search demand.
2. **word_count** — represents the amount of written content.
3. **impressions_90d** — represents observed search visibility over the 90-day window.
4. **clicks_90d** — represents observed search clicks over the 90-day window.
5. **avg_position** — represents the observed average search position.

### Label

Lane 3 is an unsupervised clustering task, so there is no predefined target label. The clustering algorithm will create groups based on similarities between content items.

### Context

I will use fields such as **content_id, content_type, main_intent, content_age_days, and freshness_tier** as contextual information when interpreting the resulting clusters. These fields are not automatically used as clustering features.

### Excluded

I will exclude **client_id** because it is an identifier rather than a content characteristic. I will also exclude provider/model identifiers and future or label-derived information from the clustering features when they are not available at the decision moment.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Query 1 — Grain verification

I will verify that the dataset has one row per content item by comparing the total number of rows with the number of unique `content_id` values.

In [8]:
# Query 1: Verify the dataset grain

total_rows = len(df)
unique_content = df["content_id"].nunique()
duplicate_count = total_rows - unique_content

print("--- Query 1: Grain Verification ---")
print("Total rows:", total_rows)
print("Unique content IDs:", unique_content)
print("Duplicate rows:", duplicate_count)

if total_rows == unique_content:
    print("PASS: one row = one content item.")
else:
    print("WARNING: duplicate content IDs are present.")



--- Query 1: Grain Verification ---
Total rows: 30000
Unique content IDs: 30000
Duplicate rows: 0
PASS: one row = one content item.


### Query 2 — Row count and observation windows

The dataset does not contain a `month` column. Instead, it provides 90-day historical metrics and two comparison windows: the last 30 days and the previous 30 days.

I will verify that these observation-window fields are present and measure how many content rows contain data for each window.

In [9]:
# Query 2: Verify row count and available observation windows

print("--- Query 2: Counts and Observation Windows ---")

print("Total content rows:", len(df))

window_groups = {
    "90-day window": [
        "impressions_90d",
        "clicks_90d",
        "sessions_90d"
    ],
    "Last 30-day window": [
        "impressions_last_30d",
        "clicks_last_30d",
        "sessions_last_30d"
    ],
    "Previous 30-day window": [
        "impressions_prev_30d",
        "clicks_prev_30d",
        "sessions_prev_30d"
    ]
}

for window_name, columns in window_groups.items():
    print(f"\n{window_name}")

    for column in columns:
        if column in df.columns:
            available = df[column].notna().sum()
            print(f"{column}: {available} rows with values")
        else:
            print(f"{column}: NOT FOUND")

--- Query 2: Counts and Observation Windows ---
Total content rows: 30000

90-day window
impressions_90d: 30000 rows with values
clicks_90d: 30000 rows with values
sessions_90d: 30000 rows with values

Last 30-day window
impressions_last_30d: 30000 rows with values
clicks_last_30d: 30000 rows with values
sessions_last_30d: 30000 rows with values

Previous 30-day window
impressions_prev_30d: 30000 rows with values
clicks_prev_30d: 30000 rows with values
sessions_prev_30d: 30000 rows with values


### Query 3 — Feature availability and missing values

I will check how many rows have available values for each of my five clustering features. This helps determine whether the selected features are sufficiently populated for the next modeling stage.

In [11]:
# Query 3: Check feature availability and missing values

selected_features = [
    "search_volume",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "avg_position"
]

print("--- Query 3: Feature Availability ---")

for column in selected_features:
    available = df[column].notna().sum()
    missing = df[column].isna().sum()

    print(f"\n{column}")
    print("Available rows:", available)
    print("Missing rows:", missing)

--- Query 3: Feature Availability ---

search_volume
Available rows: 27532
Missing rows: 2468

word_count
Available rows: 22301
Missing rows: 7699

impressions_90d
Available rows: 30000
Missing rows: 0

clicks_90d
Available rows: 30000
Missing rows: 0

avg_position
Available rows: 30000
Missing rows: 0


### Five features and why they are available

- **search_volume:** Knowable from observed search-demand data available for the content.
- **word_count:** Knowable directly from the content at the decision moment.
- **impressions_90d:** Knowable from historical search-performance data for the 90-day observation window.
- **clicks_90d:** Knowable from historical search-performance data for the 90-day observation window.
- **avg_position:** Knowable from observed search-ranking data for the relevant observation window.

These features describe measurable characteristics and historical performance that can be used to compare content items for clustering.

In [12]:
# Build the five-feature frame for Lane 3

selected_features = [
    "search_volume",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "avg_position"
]

feature_df = df[selected_features].copy()

print("--- Lane 3 Five-Feature Frame ---")
print("Number of rows:", len(feature_df))
print("Number of features:", len(feature_df.columns))

print("\nSelected features:")
print(feature_df.columns.tolist())

print("\nFirst five rows:")
display(feature_df.head())

print("\nMissing values:")
print(feature_df.isna().sum())

--- Lane 3 Five-Feature Frame ---
Number of rows: 30000
Number of features: 5

Selected features:
['search_volume', 'word_count', 'impressions_90d', 'clicks_90d', 'avg_position']

First five rows:


,search_volume,word_count,impressions_90d,clicks_90d,avg_position
0,10.0,3221.0,3803,29,10.6
1,90.0,2481.0,15320,7,20.3
2,0.0,3515.0,12581,11,36.5
3,10.0,NaN,11751,58,6.2
4,0.0,2803.0,19140,24,44.0



Missing values:
search_volume      2468
word_count         7699
impressions_90d       0
clicks_90d            0
avg_position          0
dtype: int64


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

This dataset contains observational search, content, and engagement information. It can help identify patterns and groups of similar content, but it cannot prove that one content characteristic causes another outcome.

The dataset also provides different observation windows, including 90-day and 30-day periods. These windows can overlap in time, so they should not be treated as completely independent observations.

The available history may also be unbalanced across content items. Some content may have more complete historical information than other content.

Therefore, any clusters or patterns discovered in Lane 3 should be treated as **observed, directional, and decision-support information**, rather than causal proof.

In [13]:
# Check data limitations through missing-value counts

print("--- Data Limits Check ---")

print("Total rows:", len(df))
print("Unique content items:", df["content_id"].nunique())

print("\nMissing values across selected features:")

selected_features = [
    "search_volume",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "avg_position"
]

for column in selected_features:
    missing = df[column].isna().sum()
    percentage = (missing / len(df)) * 100

    print(f"{column}: {missing} missing ({percentage:.2f}%)")

--- Data Limits Check ---
Total rows: 30000
Unique content items: 30000

Missing values across selected features:
search_volume: 2468 missing (8.23%)
word_count: 7699 missing (25.66%)
impressions_90d: 0 missing (0.00%)
clicks_90d: 0 missing (0.00%)
avg_position: 0 missing (0.00%)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.